# KR1 — Plan Freeze Rate (Estabilidade do Planejamento)

**Objetivo**: Medir que % do plano original de cada ciclo sobreviveu sem alteração interna da Insider.

**Fórmula**: `KR1 = 1 - (Σ volume_alterado_interno / Σ volume_original)`

**Reason Codes Internos**: INT_DATE (mudança de dt_planned), INT_CANCEL (cancelamento In Season), INT_GRADE (mudança de grade)

**Reason Codes Externos** (acompanhamento): EXT_CANCEL (cancelamento por outro motivo), EXT_DATE_REV (fornecedor mudou dt_reviewed)

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import datetime

ModuleNotFoundError: No module named 'pandas'

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="insider-data-lake")

today = pd.to_datetime("today").strftime("%Y%m%d")

def query_to_dataframe(query):
    """Executa query no BigQuery e retorna DataFrame."""
    query_job = client.query(query)
    results = query_job.result()
    return results.to_dataframe()

def read_sql_file(file_path):
    """Lê arquivo SQL e retorna string."""
    with open(file_path, 'r') as file:
        return file.read()

def convert_sql_to_df(file_path, **params):
    """Lê SQL, substitui parâmetros e executa."""
    sql = read_sql_file(file_path)
    if params:
        sql = sql.format(**params)
    return query_to_dataframe(sql)

/opt/anaconda3/envs/insider-ops/lib/python3.12/site-packages/google/auth/_default.py:114: UserWarning:

Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 



# 1. Carregamento dos Dados

In [ ]:
sql_path = '../1_Inputs/1_SQL/'

# Query principal: plano original vs estado atual com flags INT/EXT
df_plano = convert_sql_to_df(sql_path + 'plano_vs_atual.sql')
print(f"Plano vs Atual: {df_plano.shape[0]:,} linhas (OP-SKU), {df_plano['cycle_name'].nunique()} ciclos")
df_plano.head()

/opt/anaconda3/envs/insider-ops/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning:

BigQuery Storage module not found, fetch data with the REST endpoint instead.



Plano vs Atual: 54,570 linhas (OP-SKU), 210 ciclos


,op_code,product_sku,cycle_name,supplier_id,supplier_name,product_name,product_color,product_size,is_finished_product_order,production_order_type,...,is_int_date,is_int_cancel,is_int_grade,is_int_any,is_ext_cancel,is_ext_date_rev,is_ext_any,delta_days_planned,delta_days_reviewed,delta_planned_qty
0,OP0450023,102010100104,00,45,JKM,Tech T-shirt Gola U Masculino,Preto,P,False,committed,...,False,False,False,False,False,False,False,0,0,0
1,OP0450023,102010100105,00,45,JKM,Tech T-shirt Gola U Masculino,Preto,M,False,committed,...,False,False,False,False,False,False,False,0,0,0
2,OP0450023,102010100106,00,45,JKM,Tech T-shirt Gola U Masculino,Preto,G,False,committed,...,False,False,False,False,False,False,False,0,0,0
3,OP0450023,102010100107,00,45,JKM,Tech T-shirt Gola U Masculino,Preto,GG,False,committed,...,False,False,False,False,False,False,False,0,0,0
4,OP0450023,102010100108,00,45,JKM,Tech T-shirt Gola U Masculino,Preto,XGG,False,committed,...,False,False,False,False,False,False,False,0,0,0


In [ ]:
# Frequência de revisões por OP-SKU
df_freq = convert_sql_to_df(sql_path + 'frequencia_revisoes.sql')
print(f"Frequência de revisões: {df_freq.shape[0]:,} linhas (OP-SKU)")
df_freq.head()

/opt/anaconda3/envs/insider-ops/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning:

BigQuery Storage module not found, fetch data with the REST endpoint instead.



Frequência de revisões: 62,488 linhas (OP-SKU)


,op_code,product_sku,cycle_name,supplier_name,product_name,cycle_type,n_rev_planned,n_rev_reviewed_ext,n_rev_grade,n_rev_total,total_magnitude_dt_planned,total_magnitude_dt_reviewed,total_abs_delta_grade,first_revision_date,last_revision_date
0,OPF60N259,None,24-10-s4,DALOP,Action Top Feminino,Extra,0,0,675,675,0,0,49840,2025-11-10,2026-04-22
1,OPF61N9,None,24-07-m1,N8,NoHo Socks,Extra,0,0,231,231,0,0,28413,2025-11-10,2026-04-22
2,OPF61N20,None,24-10-s3,N8,NoHo Socks,Extra,0,0,231,231,0,0,231000,2025-11-10,2026-04-22
3,OPF61N13,None,24-09-m1,N8,NoHo Socks,Extra,0,0,229,229,0,0,93890,2025-11-10,2026-04-22
4,OPF61N10,None,24-07-m1,N8,NoHo Socks,Extra,0,0,223,223,0,0,27429,2025-11-10,2026-04-22


# 2. Cálculo do KR1 por Coorte

Para cada `cycle_name`, calcula:
- **volume_original**: soma de `baseline_planned_qty`
- **volume_alterado**: soma de `baseline_planned_qty` onde `is_int_any = True`
- **KR1** = 1 - (volume_alterado / volume_original)

Separado em KR1a (ciclos base) e KR1b (extras).

In [ ]:
# --- Ciclos excluídos das análises ---
# C012026 foi completamente cancelado; usar str.contains para pegar variantes (C012026B, etc.)
CICLOS_EXCLUIDOS = ['C012026']
mask_excluir = df_plano['cycle_name'].isin(CICLOS_EXCLUIDOS)
df_plano = df_plano[~mask_excluir].copy()
df_freq  = df_freq[~df_freq['cycle_name'].isin(CICLOS_EXCLUIDOS)].copy()
print(f"Ciclos excluídos: {CICLOS_EXCLUIDOS}")
print(f"df_plano após exclusão: {df_plano.shape[0]:,} linhas, {df_plano['cycle_name'].nunique()} ciclos")

# --- KR1 por coorte ---

kr1_coorte = df_plano.groupby(['cycle_name', 'cycle_type']).agg(
    vol_original=('baseline_planned_qty', 'sum'),
    vol_int_date=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_int_date']].sum()),
    vol_int_cancel=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_int_cancel']].sum()),
    vol_int_grade=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_int_grade']].sum()),
    vol_int_any=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_int_any']].sum()),
    vol_ext_cancel=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_ext_cancel']].sum()),
    vol_ext_date_rev=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_ext_date_rev']].sum()),
    vol_ext_any=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_ext_any']].sum()),
    n_ops=('op_code', 'nunique'),
    n_skus=('product_sku', 'nunique'),
    baseline_date=('baseline_date', 'first'),
).reset_index()

# Calcular KR1
kr1_coorte['kr1'] = 1 - (kr1_coorte['vol_int_any'] / kr1_coorte['vol_original'])
kr1_coorte['kr1_pct'] = (kr1_coorte['kr1'] * 100).round(1)

# Pct de cada reason code sobre volume alterado
for rc in ['int_date', 'int_cancel', 'int_grade']:
    kr1_coorte[f'pct_{rc}'] = (kr1_coorte[f'vol_{rc}'] / kr1_coorte['vol_original'] * 100).round(1)

# Mês-alvo: mês com maior volume planejado no baseline
mes_alvo = df_plano.groupby('cycle_name').apply(
    lambda g: g.groupby(pd.to_datetime(g['baseline_dt_planned']).dt.to_period('M'))['baseline_planned_qty']
    .sum().idxmax()
).reset_index()
mes_alvo.columns = ['cycle_name', 'mes_alvo']

kr1_coorte = kr1_coorte.merge(mes_alvo, on='cycle_name', how='left')

# Filtrar apenas coortes com mês-alvo >= nov/2025
kr1_coorte = kr1_coorte[kr1_coorte['mes_alvo'] >= pd.Period('2025-11', freq='M')]
print(f"Coortes após filtro (mes_alvo >= 2025-11): {len(kr1_coorte)}")

# Filtrar df_plano e df_freq para manter apenas os ciclos válidos
ciclos_validos = kr1_coorte['cycle_name'].unique()
df_plano = df_plano[df_plano['cycle_name'].isin(ciclos_validos)]
df_freq = df_freq[df_freq['cycle_name'].isin(ciclos_validos)]
print(f"df_plano filtrado: {df_plano.shape[0]:,} linhas")
print(f"df_freq filtrado: {df_freq.shape[0]:,} linhas")

# Ordenar por mês-alvo e volume
kr1_coorte = kr1_coorte.sort_values(['mes_alvo', 'vol_original'], ascending=[True, False])

print(f"\nTotal de coortes: {len(kr1_coorte)}")
print(f"  Base: {(kr1_coorte['cycle_type'] == 'Base').sum()}")
print(f"  Extra: {(kr1_coorte['cycle_type'] == 'Extra').sum()}")
kr1_coorte[['cycle_name', 'cycle_type', 'mes_alvo', 'vol_original', 'vol_int_any', 'kr1_pct', 'n_ops']].head(20)

Ciclos excluídos: ['C012026']
df_plano após exclusão: 53,643 linhas, 209 ciclos
Coortes após filtro (mes_alvo >= 2025-11): 127
df_plano filtrado: 15,521 linhas
df_freq filtrado: 20,174 linhas

Total de coortes: 127
  Base: 11
  Extra: 116


/var/folders/4n/1_qvk_9s5vz7pdv4bcz13z4w0000gn/T/ipykernel_64846/3029565766.py:35: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,cycle_name,cycle_type,mes_alvo,vol_original,vol_int_any,kr1_pct,n_ops
62,C112025,Base,2025-11,709871,59674,91.6,397
140,EPA2706,Extra,2025-11,75678,0,100.0,49
176,LAN1509,Extra,2025-11,73561,0,100.0,22
191,NEXTECH,Extra,2025-11,40014,0,100.0,45
92,EPA08TPT,Extra,2025-11,29998,0,100.0,8
166,LAN0210,Extra,2025-11,24583,0,100.0,55
153,FUTURE FORM 2,Extra,2025-11,17961,0,100.0,20
143,EPA2808,Extra,2025-11,17799,0,100.0,4
46,Alocação extra Adapt Fit,Extra,2025-11,17600,250,98.6,29
68,Café lab,Extra,2025-11,16091,4347,73.0,39


In [ ]:
kr1_coorte[['cycle_name', 'cycle_type', 'mes_alvo', 'vol_original', 'vol_int_any', 'kr1_pct', 'n_ops']]

,cycle_name,cycle_type,mes_alvo,vol_original,vol_int_any,kr1_pct,n_ops
62,C112025,Base,2025-11,709871,59674,91.6,397
140,EPA2706,Extra,2025-11,75678,0,100.0,49
176,LAN1509,Extra,2025-11,73561,0,100.0,22
191,NEXTECH,Extra,2025-11,40014,0,100.0,45
92,EPA08TPT,Extra,2025-11,29998,0,100.0,8
...,...,...,...,...,...,...,...
139,EPA2703,Extra,2026-06,1134,0,100.0,1
130,EPA2303,Extra,2026-06,196,0,100.0,1
58,C072026,Base,2026-07,137051,0,100.0,134
175,LAN1504,Extra,2026-07,4062,0,100.0,5


# 3. KR1 Consolidado (KR1a + KR1b + Total)

- **KR1a** (ciclos base): meta ≥ 85%
- **KR1b** (extras): meta ≥ 70%
- **KR1 consolidado**: média ponderada por volume

In [ ]:
# --- KR1 consolidado por mês-alvo e tipo de ciclo ---

# Filtrar apenas coortes com mes_alvo >= mês atual (OKR ativo)
mes_atual = pd.Period(pd.Timestamp.today(), freq='M')
kr1_coorte_okr = kr1_coorte[kr1_coorte['mes_alvo'] >= mes_atual]
print(f"Coortes no OKR ativo (mes_alvo >= {mes_atual}): {len(kr1_coorte_okr)}")

# Por tipo de ciclo (KR1a e KR1b)
kr1_tipo = kr1_coorte_okr.groupby('cycle_type').agg(
    vol_original=('vol_original', 'sum'),
    vol_int_any=('vol_int_any', 'sum'),
    vol_ext_any=('vol_ext_any', 'sum'),
).reset_index()
kr1_tipo['kr1'] = 1 - (kr1_tipo['vol_int_any'] / kr1_tipo['vol_original'])
kr1_tipo['kr1_pct'] = (kr1_tipo['kr1'] * 100).round(1)
kr1_tipo['meta'] = kr1_tipo['cycle_type'].map({'Base': 85.0, 'Extra': 70.0})

print("=== KR1 por Tipo de Ciclo ===")
print(kr1_tipo[['cycle_type', 'vol_original', 'vol_int_any', 'kr1_pct', 'meta']].to_string(index=False))

# Consolidado total
vol_total = kr1_tipo['vol_original'].sum()
vol_alt_total = kr1_tipo['vol_int_any'].sum()
kr1_total = (1 - vol_alt_total / vol_total) * 100

print(f"\n=== KR1 Consolidado (OKR ativo): {kr1_total:.1f}% ===")
print(f"Volume original total: {vol_total:,.0f}")
print(f"Volume alterado (INT): {vol_alt_total:,.0f}")

Coortes no OKR ativo (mes_alvo >= 2026-04): 39
=== KR1 por Tipo de Ciclo ===
cycle_type  vol_original  vol_int_any  kr1_pct  meta
      Base       1115132       704215     36.8  85.0
     Extra        255059       143072     43.9  70.0

=== KR1 Consolidado (OKR ativo): 38.2% ===
Volume original total: 1,370,191
Volume alterado (INT): 847,287


In [ ]:
# --- KR1 por mês-alvo (para visão temporal) ---

kr1_mes = kr1_coorte.groupby(['mes_alvo', 'cycle_type']).agg(
    vol_original=('vol_original', 'sum'),
    vol_int_any=('vol_int_any', 'sum'),
    vol_int_date=('vol_int_date', 'sum'),
    vol_int_cancel=('vol_int_cancel', 'sum'),
    vol_int_grade=('vol_int_grade', 'sum'),
    vol_ext_any=('vol_ext_any', 'sum'),
    vol_ext_cancel=('vol_ext_cancel', 'sum'),
    vol_ext_date_rev=('vol_ext_date_rev', 'sum'),
    n_coortes=('cycle_name', 'nunique'),
).reset_index()

kr1_mes['kr1_pct'] = ((1 - kr1_mes['vol_int_any'] / kr1_mes['vol_original']) * 100).round(1)
kr1_mes['mes_alvo_str'] = kr1_mes['mes_alvo'].astype(str)

# Consolidado por mês (ambos tipos juntos)
kr1_mes_total = kr1_coorte.groupby('mes_alvo').agg(
    vol_original=('vol_original', 'sum'),
    vol_int_any=('vol_int_any', 'sum'),
).reset_index()
kr1_mes_total['kr1_pct'] = ((1 - kr1_mes_total['vol_int_any'] / kr1_mes_total['vol_original']) * 100).round(1)
kr1_mes_total['mes_alvo_str'] = kr1_mes_total['mes_alvo'].astype(str)

print("=== KR1 por Mês-Alvo ===")
print(kr1_mes[['mes_alvo_str', 'cycle_type', 'vol_original', 'kr1_pct', 'n_coortes']].to_string(index=False))

=== KR1 por Mês-Alvo ===
mes_alvo_str cycle_type  vol_original  kr1_pct  n_coortes
     2025-11       Base        709871     91.6          1
     2025-11      Extra        377645     96.9         18
     2025-12       Base        536059     94.6          1
     2025-12      Extra        218123     79.5         17
     2026-01       Base        396201     89.8          3
     2026-01      Extra        237276     58.7         14
     2026-02       Base        361086     82.5          1
     2026-02      Extra         91526     85.8         13
     2026-03       Base        299397     62.7          1
     2026-03      Extra        310910     77.2         19
     2026-04       Base        356484     29.7          1
     2026-04      Extra        147992     45.8         18
     2026-05       Base        282514     22.4          1
     2026-05      Extra         62637     34.8          7
     2026-06       Base        339083     30.9          1
     2026-06      Extra         36462     39.7 

# 4. Visualizações — Visão de Planejamento

In [ ]:
# --- Gráfico 1: KR1 por Mês-Alvo (bar chart, base vs extras, com metas) ---

colors_cycle = {'Base': '#1f77b4', 'Extra': '#ff7f0e'}

fig1 = px.bar(
    kr1_mes,
    x='mes_alvo_str',
    y='kr1_pct',
    color='cycle_type',
    barmode='group',
    color_discrete_map=colors_cycle,
    labels={'kr1_pct': 'KR1 (%)', 'mes_alvo_str': 'Mês-Alvo', 'cycle_type': 'Tipo de Ciclo'},
    title='KR1 — Plan Freeze Rate por Mês-Alvo',
    text='kr1_pct',
)

# Linhas de meta
fig1.add_hline(y=85, line_dash="dash", line_color="#1f77b4", opacity=0.5,
               annotation_text="Meta Base (85%)", annotation_position="top left")
fig1.add_hline(y=70, line_dash="dash", line_color="#ff7f0e", opacity=0.5,
               annotation_text="Meta Extra (70%)", annotation_position="bottom left")

fig1.update_traces(textposition='outside')
fig1.update_layout(yaxis_range=[0, 105], template='plotly_white')
fig1.show()

In [ ]:
# --- Gráfico 2: Waterfall de Alterações por Mês-Alvo ---
# Exclusão mútua: prioridade CANCEL > DATE > GRADE (cada OP-SKU aparece em apenas 1 barra)

# Filtro de tipo de ciclo para o waterfall (altere para 'Extra' ou None para ver todos)
WATERFALL_CYCLE_TYPE = 'Base'  # 'Base' | 'Extra' | None

# Classificar cada OP-SKU com reason code exclusivo
df_plano['int_reason_excl'] = np.where(
    df_plano['is_int_cancel'], 'INT_CANCEL',
    np.where(
        df_plano['is_int_date'], 'INT_DATE',
        np.where(
            df_plano['is_int_grade'], 'INT_GRADE',
            'Inalterado'
        )
    )
)

# Merge com mes_alvo para agrupar (cycle_type já existe em df_plano)
df_plano_mes = df_plano.merge(
    kr1_coorte[['cycle_name', 'mes_alvo']].drop_duplicates(),
    on='cycle_name', how='left'
)

# Aplicar filtro de tipo de ciclo
if WATERFALL_CYCLE_TYPE:
    df_plano_mes = df_plano_mes[df_plano_mes['cycle_type'] == WATERFALL_CYCLE_TYPE]

# Volume exclusivo por mês-alvo e reason code
wf_excl = df_plano_mes.groupby(['mes_alvo', 'int_reason_excl'])['baseline_planned_qty'].sum().reset_index()
wf_excl = wf_excl.pivot(index='mes_alvo', columns='int_reason_excl', values='baseline_planned_qty').fillna(0)

# Garantir colunas existem
for col in ['INT_CANCEL', 'INT_DATE', 'INT_GRADE', 'Inalterado']:
    if col not in wf_excl.columns:
        wf_excl[col] = 0

waterfall_data = []
for mes_alvo in wf_excl.index:
    mes_str = str(mes_alvo)
    vol_orig = wf_excl.loc[mes_alvo].sum()
    v_cancel = wf_excl.loc[mes_alvo, 'INT_CANCEL']
    v_date = wf_excl.loc[mes_alvo, 'INT_DATE']
    v_grade = wf_excl.loc[mes_alvo, 'INT_GRADE']
    v_unchanged = wf_excl.loc[mes_alvo, 'Inalterado']

    waterfall_data.append({'Mês': mes_str, 'Etapa': 'Volume Original', 'Valor': vol_orig, 'Tipo': 'absolute'})
    waterfall_data.append({'Mês': mes_str, 'Etapa': '- INT_CANCEL', 'Valor': -v_cancel, 'Tipo': 'relative'})
    waterfall_data.append({'Mês': mes_str, 'Etapa': '- INT_DATE', 'Valor': -v_date, 'Tipo': 'relative'})
    waterfall_data.append({'Mês': mes_str, 'Etapa': '- INT_GRADE', 'Valor': -v_grade, 'Tipo': 'relative'})
    waterfall_data.append({'Mês': mes_str, 'Etapa': 'Inalterado', 'Valor': v_unchanged, 'Tipo': 'total'})

df_waterfall = pd.DataFrame(waterfall_data)

mes_escolhido = '2026-06'
wf_mes = df_waterfall[df_waterfall['Mês'] == mes_escolhido]

tipo_label = WATERFALL_CYCLE_TYPE if WATERFALL_CYCLE_TYPE else 'Todos'

fig2 = go.Figure(go.Waterfall(
    name=mes_escolhido,
    orientation="v",
    measure=wf_mes['Tipo'].tolist(),
    x=wf_mes['Etapa'].tolist(),
    y=wf_mes['Valor'].tolist(),
    textposition="outside",
    text=[f"{v:,.0f}" for v in wf_mes['Valor']],
    connector={"line": {"color": "rgb(63, 63, 63)"}},
    increasing={"marker": {"color": "#2ca02c"}},
    decreasing={"marker": {"color": "#d62728"}},
    totals={"marker": {"color": "#1f77b4"}},
))

fig2.update_layout(
    title=f"Waterfall de Alterações Internas — {mes_escolhido} ({tipo_label})<br><sup>Exclusão mútua: CANCEL > DATE > GRADE</sup>",
    yaxis_title="Volume (peças)",
    template='plotly_white',
    showlegend=False,
)
fig2.show()

In [ ]:
# --- Gráfico 3: Reason Code Breakdown (INT + EXT lado a lado) — Stacked Bar por Mês (apenas ciclos Base) ---
# Nota: as barras INT podem somar > total real pois uma OP pode ter INT_DATE + INT_GRADE
# simultaneamente. A linha pontilhada mostra o % total correto (sem double-count).

colors_rc = {
    'INT_DATE': '#d62728',
    'INT_CANCEL': '#e377c2',
    'INT_GRADE': '#ff7f0e',
    'EXT_CANCEL': '#9467bd',
    'EXT_DATE_REV': '#8c564b',
}

rc_cols_int = {'vol_int_date': 'INT_DATE', 'vol_int_cancel': 'INT_CANCEL', 'vol_int_grade': 'INT_GRADE'}
rc_cols_ext = {'vol_ext_cancel': 'EXT_CANCEL', 'vol_ext_date_rev': 'EXT_DATE_REV'}
all_rc = {**rc_cols_int, **rc_cols_ext}

# Filtrar apenas ciclos Base
kr1_mes_base = kr1_mes[kr1_mes['cycle_type'] == 'Base'].copy()

rc_data = []
for _, row in kr1_mes_base.iterrows():
    for col, label in all_rc.items():
        rc_data.append({
            'Mês-Alvo': row['mes_alvo_str'],
            'Tipo Ciclo': row['cycle_type'],
            'Reason Code': label,
            'Volume': row[col],
            'Origem': 'Interno' if label.startswith('INT') else 'Externo',
        })

df_rc = pd.DataFrame(rc_data)

# % sobre volume original do mês (normalização) — Base apenas
vol_mes_base = kr1_mes_base.groupby('mes_alvo_str', as_index=False).agg(
    vol_original=('vol_original', 'sum'),
    vol_int_any=('vol_int_any', 'sum'),
    vol_ext_any=('vol_ext_any', 'sum'),
)
vol_mes_map = vol_mes_base.set_index('mes_alvo_str')['vol_original'].to_dict()

df_rc['pct_vol_original'] = df_rc.apply(
    lambda r: r['Volume'] / vol_mes_map[r['Mês-Alvo']] * 100 if vol_mes_map.get(r['Mês-Alvo'], 0) > 0 else 0,
    axis=1
)

# % total alterado real (INT) por mês — Base apenas
pct_total_int = vol_mes_base.copy()
pct_total_int['pct_alterado'] = (pct_total_int['vol_int_any'] / pct_total_int['vol_original'] * 100).round(1)

# % externo total real por mês — Base apenas
vol_ext_mes = vol_mes_base.copy()
vol_ext_mes['pct_ext_alterado'] = (vol_ext_mes['vol_ext_any'] / vol_ext_mes['vol_original'] * 100).round(1)

fig3 = px.bar(
    df_rc,
    x='Mês-Alvo',
    y='pct_vol_original',
    color='Reason Code',
    color_discrete_map=colors_rc,
    facet_col='Origem',
    labels={'pct_vol_original': '% do Volume Original', 'Mês-Alvo': 'Mês-Alvo'},
    title='Breakdown de Reason Codes (Ciclos Base) — % do Volume Original por Mês'
          '<br><sup>Barras podem somar > total real (overlap). Linha = % total sem double-count.</sup>',
    barmode='stack',
)

# Adicionar linha de % total real (INT) no facet "Interno"
fig3.add_trace(
    go.Scatter(
        x=pct_total_int['mes_alvo_str'],
        y=pct_total_int['pct_alterado'],
        mode='lines+markers+text',
        name='% Total Alterado (INT)',
        text=[f"{v:.1f}%" for v in pct_total_int['pct_alterado']],
        textposition='top center',
        line=dict(color='black', width=2, dash='dot'),
        marker=dict(size=7, color='black'),
        showlegend=True,
    ),
    row=1, col=1,
)

# Adicionar linha de % total real (EXT) no facet "Externo"
fig3.add_trace(
    go.Scatter(
        x=vol_ext_mes['mes_alvo_str'],
        y=vol_ext_mes['pct_ext_alterado'],
        mode='lines+markers+text',
        name='% Total Alterado (EXT)',
        text=[f"{v:.1f}%" for v in vol_ext_mes['pct_ext_alterado']],
        textposition='top center',
        line=dict(color='black', width=2, dash='dot'),
        marker=dict(size=7, color='black'),
        showlegend=True,
    ),
    row=1, col=2,
)

fig3.update_layout(template='plotly_white')
fig3.show()

In [ ]:
# --- Gráfico 4: KR1 por Coorte Individual (top 20 por volume) ---

top_coortes = kr1_coorte.nlargest(20, 'vol_original').copy()
top_coortes['mes_alvo_str'] = top_coortes['mes_alvo'].astype(str)

fig4 = px.bar(
    top_coortes,
    x='cycle_name',
    y='kr1_pct',
    color='cycle_type',
    color_discrete_map=colors_cycle,
    labels={'kr1_pct': 'KR1 (%)', 'cycle_name': 'Ciclo', 'cycle_type': 'Tipo'},
    title='KR1 por Coorte — Top 20 por Volume',
    text='kr1_pct',
    hover_data=['vol_original', 'vol_int_any', 'n_ops', 'mes_alvo_str'],
)

fig4.add_hline(y=85, line_dash="dash", line_color="#1f77b4", opacity=0.5)
fig4.add_hline(y=70, line_dash="dash", line_color="#ff7f0e", opacity=0.5)
fig4.update_traces(textposition='outside')
fig4.update_layout(yaxis_range=[0, 105], template='plotly_white', xaxis_tickangle=-45)
fig4.show()

# 5. Análise de Frequência de Revisões

In [ ]:
# --- Gráfico 5: Distribuição de Revisões por OP-SKU (histogram) ---

# Criar faixas de revisão
df_freq['faixa_rev'] = pd.cut(
    df_freq['n_rev_total'],
    bins=[-1, 0, 1, 2, 3, 5, 100],
    labels=['0', '1', '2', '3', '4-5', '6+']
)

# Separar INT e EXT
df_freq['n_rev_int'] = df_freq['n_rev_planned'] + df_freq['n_rev_grade']
df_freq['n_rev_ext'] = df_freq['n_rev_reviewed_ext']

freq_dist = df_freq.groupby(['faixa_rev', 'cycle_type']).agg(
    n_op_sku=('op_code', 'count'),
).reset_index()

fig5 = px.bar(
    freq_dist,
    x='faixa_rev',
    y='n_op_sku',
    color='cycle_type',
    color_discrete_map=colors_cycle,
    barmode='group',
    labels={'faixa_rev': 'Nº de Revisões', 'n_op_sku': 'Qtd OP-SKUs', 'cycle_type': 'Tipo'},
    title='Distribuição de Frequência de Revisões por OP-SKU',
    text='n_op_sku',
)

fig5.update_traces(textposition='outside')
fig5.update_layout(template='plotly_white')
fig5.show()

# Resumo
pct_sem_rev = (df_freq['n_rev_total'] == 0).mean() * 100
pct_3_mais = (df_freq['n_rev_total'] >= 3).mean() * 100
print(f"\n{pct_sem_rev:.1f}% das OP-SKUs nunca tiveram revisão")
print(f"{pct_3_mais:.1f}% tiveram 3+ revisões")

/var/folders/4n/1_qvk_9s5vz7pdv4bcz13z4w0000gn/T/ipykernel_64846/785512017.py:14: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




44.8% das OP-SKUs nunca tiveram revisão
27.7% tiveram 3+ revisões


In [ ]:
# --- Gráfico 6: Heatmap Revisões × Fornecedor ---
# Top 15 fornecedores por volume de OP-SKUs

top_suppliers = df_freq.groupby('supplier_name')['op_code'].count().nlargest(15).index

df_freq_top = df_freq[df_freq['supplier_name'].isin(top_suppliers)].copy()
df_freq_top['faixa_rev_ext'] = pd.cut(
    df_freq_top['n_rev_ext'],
    bins=[-1, 0, 1, 2, 100],
    labels=['0', '1', '2', '3+']
)

heatmap_data = df_freq_top.groupby(['supplier_name', 'faixa_rev_ext']).size().reset_index(name='count')
heatmap_pivot = heatmap_data.pivot(index='supplier_name', columns='faixa_rev_ext', values='count').fillna(0)

# Normalizar por linha (cada fornecedor soma 100%)
heatmap_pct = heatmap_pivot.div(heatmap_pivot.sum(axis=1), axis=0) * 100

fig6 = px.imshow(
    heatmap_pct.values,
    x=heatmap_pct.columns.tolist(),
    y=heatmap_pct.index.tolist(),
    labels=dict(x='Revisões Externas (EXT_DATE_REV)', y='Fornecedor', color='% OP-SKUs'),
    title='Heatmap: Revisões Externas por Fornecedor (% por linha)',
    color_continuous_scale='YlOrRd',
    text_auto='.1f',
    aspect='auto',
    zmin=0,
    zmax=100,
)
fig6.update_layout(template='plotly_white')
fig6.show()

/var/folders/4n/1_qvk_9s5vz7pdv4bcz13z4w0000gn/T/ipykernel_64846/626037793.py:13: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [ ]:
df_plano

,op_code,product_sku,cycle_name,supplier_id,supplier_name,product_name,product_color,product_size,is_finished_product_order,production_order_type,...,is_int_cancel,is_int_grade,is_int_any,is_ext_cancel,is_ext_date_rev,is_ext_any,delta_days_planned,delta_days_reviewed,delta_planned_qty,int_reason_excl
31764,OPF101N16,203120720104,Alocação extra Adapt Fit,101,CLARA BELLA,Stirrup Legging Feminino,Preto,P,True,committed,...,False,False,False,False,False,False,0,0,0,Inalterado
31765,OPF101N16,203120720105,Alocação extra Adapt Fit,101,CLARA BELLA,Stirrup Legging Feminino,Preto,M,True,committed,...,False,False,False,False,False,False,0,0,0,Inalterado
31766,OPF101N16,203120720106,Alocação extra Adapt Fit,101,CLARA BELLA,Stirrup Legging Feminino,Preto,G,True,committed,...,False,False,False,False,False,False,0,0,0,Inalterado
31767,OPF101N16,203120720107,Alocação extra Adapt Fit,101,CLARA BELLA,Stirrup Legging Feminino,Preto,GG,True,committed,...,False,False,False,False,False,False,0,0,0,Inalterado
31768,OPF101N17,203120720104,Alocação extra Adapt Fit,101,CLARA BELLA,Stirrup Legging Feminino,Preto,P,True,committed,...,False,False,False,False,False,False,0,0,0,Inalterado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54543,OPF102N69,2095821010304,Whateverproof / LifeProof,102,BLUTEXTIL,HighTech T-shirt LifeProof Feminino,Azul Marinho,P,False,committed,...,False,False,False,True,False,True,0,0,0,Inalterado
54544,OPF102N69,2095821010305,Whateverproof / LifeProof,102,BLUTEXTIL,HighTech T-shirt LifeProof Feminino,Azul Marinho,M,False,committed,...,False,False,False,True,False,True,0,0,0,Inalterado
54545,OPF102N69,2095821010306,Whateverproof / LifeProof,102,BLUTEXTIL,HighTech T-shirt LifeProof Feminino,Azul Marinho,G,False,committed,...,False,False,False,True,False,True,0,0,0,Inalterado
54546,OPF102N69,2095821010307,Whateverproof / LifeProof,102,BLUTEXTIL,HighTech T-shirt LifeProof Feminino,Azul Marinho,GG,False,committed,...,False,False,False,True,False,True,0,0,0,Inalterado


In [ ]:
# --- Gráfico 7: Scatter de Instabilidade ---
# Eixo X: nº revisões total, Eixo Y: magnitude total (dias), Tamanho: volume, Cor: INT vs EXT dominante

df_freq['dominant_type'] = np.where(
    df_freq['n_rev_int'] >= df_freq['n_rev_ext'],
    'Interno (INT)',
    'Externo (EXT)'
)
df_freq['total_magnitude'] = df_freq['total_magnitude_dt_planned'] + df_freq['total_magnitude_dt_reviewed']

# Filtrar apenas quem teve revisão
df_freq_rev = df_freq[df_freq['n_rev_total'] > 0].copy()

# Juntar com plano para ter o volume (baseline_planned_qty)
df_freq_rev_vol = df_freq_rev.merge(
    df_plano.groupby(['op_code', 'product_sku'])['baseline_planned_qty'].first().reset_index(),
    on=['op_code', 'product_sku'],
    how='left'
)
# Preencher NaN de volume (OP-SKUs sem match no df_plano) com 0
df_freq_rev_vol['baseline_planned_qty'] = df_freq_rev_vol['baseline_planned_qty'].fillna(0)

fig7 = px.scatter(
    df_freq_rev_vol,
    x='n_rev_total',
    y='total_magnitude',
    size='baseline_planned_qty',
    color='dominant_type',
    color_discrete_map={'Interno (INT)': '#d62728', 'Externo (EXT)': '#9467bd'},
    labels={
        'n_rev_total': 'Nº Total de Revisões',
        'total_magnitude': 'Magnitude Total (dias)',
        'baseline_planned_qty': 'Volume Planejado',
        'dominant_type': 'Origem Dominante',
    },
    title='Scatter de Instabilidade — Frequência × Magnitude',
    hover_data=['op_code', 'product_sku', 'supplier_name', 'cycle_name'],
    size_max=30,
    opacity=0.6,
)

fig7.update_layout(template='plotly_white')
fig7.show()

# OPs mais instáveis
print("\n=== Top 10 OP-SKUs mais instáveis (por nº de revisões) ===")
print(df_freq_rev.nlargest(10, 'n_rev_total')[
    ['op_code', 'product_sku', 'cycle_name', 'supplier_name', 'n_rev_planned', 'n_rev_reviewed_ext', 'n_rev_grade', 'n_rev_total', 'total_magnitude']
].to_string(index=False))


=== Top 10 OP-SKUs mais instáveis (por nº de revisões) ===
   op_code   product_sku cycle_name supplier_name  n_rev_planned  n_rev_reviewed_ext  n_rev_grade  n_rev_total  total_magnitude
OPF37N1803  201040122605    EPA0512    BAE BRASIL              1                  29            0           30               82
OPF37N1803  201040122606    EPA0512    BAE BRASIL              1                  29            0           30               82
OPF37N1803  201040122607    EPA0512    BAE BRASIL              1                  29            0           30               82
OPF37N1873  202180687504    C042026    BAE BRASIL              1                  25            0           26               67
OPF37N1873  202180687505    C042026    BAE BRASIL              1                  25            0           26               67
OPF37N1873  202180687506    C042026    BAE BRASIL              1                  25            0           26               67
OPF37N1873  202180687507    C042026    BAE B

# 6. Drill-Down: Timeline de uma OP específica

Selecione uma OP e SKU para ver a evolução de `dt_planned`, `dt_reviewed` e `planned_quantity` ao longo dos snapshots diários.

In [ ]:
# --- Timeline de OP: selecionar a OP-SKU mais instável ---

top_instavel = df_freq_rev.nlargest(1, 'n_rev_total').iloc[0]
op_drill = top_instavel['op_code']
sku_drill = top_instavel['product_sku']

print(f"Drill-down: OP {op_drill}, SKU {sku_drill}")
print(f"  Fornecedor: {top_instavel['supplier_name']}")
print(f"  Ciclo: {top_instavel['cycle_name']}")
print(f"  Revisões: {int(top_instavel['n_rev_total'])} (INT_DATE: {int(top_instavel['n_rev_planned'])}, EXT_DATE_REV: {int(top_instavel['n_rev_reviewed_ext'])}, INT_GRADE: {int(top_instavel['n_rev_grade'])})")

# Buscar timeline via SQL parametrizado
df_timeline = convert_sql_to_df(
    sql_path + 'timeline_revisoes.sql',
    op_code=op_drill,
    product_sku=sku_drill,
)
df_timeline.head(10)

Drill-down: OP OPF37N1803, SKU 201040122605
  Fornecedor: BAE BRASIL
  Ciclo: EPA0512
  Revisões: 30 (INT_DATE: 1, EXT_DATE_REV: 29, INT_GRADE: 0)


/opt/anaconda3/envs/insider-ops/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning:

BigQuery Storage module not found, fetch data with the REST endpoint instead.



,op_code,product_sku,cycle_name,supplier_name,product_name,ingestion_date,current_production_stage,planned_quantity,dt_planned_entry_warehouse,dt_reviewed_entry_warehouse,received_quantity,cutted_quantity,prev_dt_planned,prev_dt_reviewed,prev_planned_qty,change_dt_planned,change_dt_reviewed,change_grade
0,OPF37N1803,201040122605,EPA0512,BAE BRASIL,Calcinha Safe Feminino,2025-12-13,waiting_fabric_arrival,809,2026-03-30,2026-03-30,<NA>,<NA>,NaT,NaT,<NA>,None,None,None
1,OPF37N1803,201040122605,EPA0512,BAE BRASIL,Calcinha Safe Feminino,2025-12-14,waiting_fabric_arrival,809,2026-03-30,2026-03-30,<NA>,<NA>,2026-03-30,2026-03-30,809,None,None,None
2,OPF37N1803,201040122605,EPA0512,BAE BRASIL,Calcinha Safe Feminino,2025-12-15,waiting_fabric_arrival,809,2026-03-30,2026-03-30,<NA>,<NA>,2026-03-30,2026-03-30,809,None,None,None
3,OPF37N1803,201040122605,EPA0512,BAE BRASIL,Calcinha Safe Feminino,2025-12-16,waiting_fabric_arrival,809,2026-03-30,2026-03-30,<NA>,<NA>,2026-03-30,2026-03-30,809,None,None,None
4,OPF37N1803,201040122605,EPA0512,BAE BRASIL,Calcinha Safe Feminino,2025-12-17,waiting_fabric_arrival,809,2026-03-30,2026-03-30,<NA>,<NA>,2026-03-30,2026-03-30,809,None,None,None
5,OPF37N1803,201040122605,EPA0512,BAE BRASIL,Calcinha Safe Feminino,2025-12-18,waiting_fabric_arrival,809,2026-03-30,2026-03-30,<NA>,<NA>,2026-03-30,2026-03-30,809,None,None,None
6,OPF37N1803,201040122605,EPA0512,BAE BRASIL,Calcinha Safe Feminino,2025-12-19,waiting_fabric_arrival,809,2026-03-30,2026-03-30,<NA>,<NA>,2026-03-30,2026-03-30,809,None,None,None
7,OPF37N1803,201040122605,EPA0512,BAE BRASIL,Calcinha Safe Feminino,2025-12-20,waiting_fabric_arrival,809,2026-03-30,2026-03-30,<NA>,<NA>,2026-03-30,2026-03-30,809,None,None,None
8,OPF37N1803,201040122605,EPA0512,BAE BRASIL,Calcinha Safe Feminino,2025-12-21,waiting_fabric_arrival,809,2026-03-30,2026-03-30,<NA>,<NA>,2026-03-30,2026-03-30,809,None,None,None
9,OPF37N1803,201040122605,EPA0512,BAE BRASIL,Calcinha Safe Feminino,2025-12-22,waiting_fabric_arrival,809,2026-03-30,2026-03-30,<NA>,<NA>,2026-03-30,2026-03-30,809,None,None,None


In [ ]:
# --- Gráfico 8: Timeline de datas planejadas/revisadas de uma OP ---

df_timeline['ingestion_date'] = pd.to_datetime(df_timeline['ingestion_date'])
df_timeline['dt_planned_entry_warehouse'] = pd.to_datetime(df_timeline['dt_planned_entry_warehouse'])
df_timeline['dt_reviewed_entry_warehouse'] = pd.to_datetime(df_timeline['dt_reviewed_entry_warehouse'])

fig8 = go.Figure()

# dt_planned ao longo do tempo
fig8.add_trace(go.Scatter(
    x=df_timeline['ingestion_date'],
    y=df_timeline['dt_planned_entry_warehouse'],
    mode='lines+markers',
    name='dt_planned',
    line=dict(color='#d62728', width=2),
    marker=dict(
        size=8,
        color=['#d62728' if c == 'INT_DATE' else 'rgba(0,0,0,0)' for c in df_timeline['change_dt_planned'].fillna('')],
        line=dict(width=1, color='black'),
    ),
))

# dt_reviewed ao longo do tempo
fig8.add_trace(go.Scatter(
    x=df_timeline['ingestion_date'],
    y=df_timeline['dt_reviewed_entry_warehouse'],
    mode='lines+markers',
    name='dt_reviewed',
    line=dict(color='#9467bd', width=2),
    marker=dict(
        size=8,
        color=['#9467bd' if c == 'EXT_DATE_REV' else 'rgba(0,0,0,0)' for c in df_timeline['change_dt_reviewed'].fillna('')],
        line=dict(width=1, color='black'),
    ),
))

# Marcar mudanças de grade no eixo secundário
has_grade_change = df_timeline[df_timeline['change_grade'] == 'INT_GRADE']
if len(has_grade_change) > 0:
    fig8.add_trace(go.Scatter(
        x=has_grade_change['ingestion_date'],
        y=has_grade_change['dt_planned_entry_warehouse'],
        mode='markers',
        name='INT_GRADE',
        marker=dict(size=14, color='#ff7f0e', symbol='diamond', line=dict(width=2, color='black')),
    ))

fig8.update_layout(
    title=f'Timeline de Revisões — OP {op_drill} / SKU {sku_drill}',
    xaxis_title='Data do Snapshot',
    yaxis_title='Data Planejada/Revisada',
    template='plotly_white',
    hovermode='x unified',
)
fig8.show()

In [ ]:
# --- Gráfico 9: Evolução da planned_quantity na timeline ---

fig9 = go.Figure()

fig9.add_trace(go.Scatter(
    x=df_timeline['ingestion_date'],
    y=df_timeline['planned_quantity'],
    mode='lines+markers',
    name='planned_quantity',
    line=dict(color='#ff7f0e', width=2),
    marker=dict(
        size=8,
        color=['#ff7f0e' if c == 'INT_GRADE' else 'rgba(0,0,0,0.1)' for c in df_timeline['change_grade'].fillna('')],
        line=dict(width=1, color='black'),
    ),
    fill='tozeroy',
    fillcolor='rgba(255,127,14,0.1)',
))

fig9.update_layout(
    title=f'Evolução da Grade (planned_quantity) — OP {op_drill} / SKU {sku_drill}',
    xaxis_title='Data do Snapshot',
    yaxis_title='Quantidade Planejada',
    template='plotly_white',
)
fig9.show()

# 7. Evolução Temporal do KR1

Compara cada snapshot semanal contra o baseline para visualizar como o KR1 "degrada" ao longo do tempo.
Permite identificar **quando** as mudanças acontecem e quais meses-alvo são mais impactados.

In [ ]:
# --- Carregar dados de evolução temporal ---
df_evolucao = convert_sql_to_df(sql_path + 'kr1_evolucao.sql')
df_evolucao  = df_evolucao[~df_evolucao['cycle_name'].isin(CICLOS_EXCLUIDOS)].copy()
print(f"Evolução temporal: {df_evolucao.shape[0]:,} linhas (semana × ciclo)")
print(f"Semanas: {df_evolucao['snapshot_week'].nunique()}, Ciclos: {df_evolucao['cycle_name'].nunique()}")

# Filtrar apenas ciclos válidos (mesmos do KR1 principal)
df_evolucao = df_evolucao[df_evolucao['cycle_name'].isin(ciclos_validos)]

# Merge com mes_alvo usando kr1_coorte como fonte (evita problemas com o tipo Period)
df_evolucao = df_evolucao.merge(
    kr1_coorte[['cycle_name', 'mes_alvo']].drop_duplicates(),
    on='cycle_name', how='left'
)

# KR1 por linha (semana × ciclo)
df_evolucao['kr1'] = 1 - (df_evolucao['vol_int_any'] / df_evolucao['vol_original'])
df_evolucao['kr1_pct'] = (df_evolucao['kr1'] * 100).round(1)

# Converter para string (evitar TypeError com Period/date no Plotly)
df_evolucao['snapshot_week'] = pd.to_datetime(df_evolucao['snapshot_week'])
df_evolucao['mes_alvo_str'] = df_evolucao['mes_alvo'].astype(str)

print(f"\nApós filtro: {df_evolucao.shape[0]:,} linhas, {df_evolucao['cycle_name'].nunique()} ciclos")
df_evolucao.head()

/opt/anaconda3/envs/insider-ops/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning:

BigQuery Storage module not found, fetch data with the REST endpoint instead.



Evolução temporal: 4,135 linhas (semana × ciclo)
Semanas: 24, Ciclos: 208

Após filtro: 2,167 linhas, 126 ciclos


,snapshot_week,cycle_name,cycle_type,vol_original,vol_int_date,vol_int_cancel,vol_int_grade,vol_int_any,vol_ext_cancel,vol_ext_date_rev,vol_ext_any,mes_alvo,kr1,kr1_pct,mes_alvo_str
0,2025-11-10,Alocação extra Adapt Fit,Extra,17600,0,0,0,0,0,2250,2250,2025-11,1.0,100.0,2025-11
1,2025-11-10,BOLD 2 - Revisada,Extra,84506,0,0,0,0,4480,0,4480,2026-01,1.0,100.0,2026-01
2,2025-11-10,C012026B,Base,394640,0,0,0,0,71196,23472,94668,2026-01,1.0,100.0,2026-01
3,2025-11-10,C022026,Base,361086,2042,0,1371,8992,29657,0,29657,2026-02,0.975097,97.5,2026-02
4,2025-11-10,C032026,Base,299397,0,0,0,0,0,0,0,2026-03,1.0,100.0,2026-03


In [ ]:
# --- Gráfico A: Curva de degradação do KR1 por mês-alvo × tipo de ciclo ---

# Agregar por semana × mes_alvo × cycle_type
evol_mes = df_evolucao.groupby(['snapshot_week', 'mes_alvo_str', 'cycle_type']).agg(
    vol_original=('vol_original', 'sum'),
    vol_int_any=('vol_int_any', 'sum'),
).reset_index()

evol_mes['kr1_pct'] = ((1 - evol_mes['vol_int_any'] / evol_mes['vol_original']) * 100).round(1)

fig_a = px.line(
    evol_mes.sort_values('snapshot_week'),
    x='snapshot_week', y='kr1_pct', color='mes_alvo_str',
    facet_col='cycle_type',
    title='Evolução do KR1 por Mês-Alvo — Curva de Degradação (Base vs Extra)',
    labels={'snapshot_week': 'Semana', 'kr1_pct': 'KR1 (%)', 'mes_alvo_str': 'Mês-Alvo'},
    markers=True
)
# Linha de meta por facet: Base=85%, Extra=70%
fig_a.add_hline(y=85, line_dash="dot", line_color="green", annotation_text="Meta 85%", col=1)
fig_a.add_hline(y=70, line_dash="dot", line_color="orange", annotation_text="Meta 70%", col=2)
fig_a.update_layout(yaxis_range=[0, 105], template='plotly_white')
fig_a.update_yaxes(range=[0, 105])
fig_a.show()

In [ ]:
# --- Gráfico B: KR1 consolidado ao longo do tempo — Base vs Extra ---

evol_tipo = df_evolucao.groupby(['snapshot_week', 'cycle_type']).agg(
    vol_original=('vol_original', 'sum'),
    vol_int_any=('vol_int_any', 'sum'),
).reset_index()

evol_tipo['kr1_pct'] = ((1 - evol_tipo['vol_int_any'] / evol_tipo['vol_original']) * 100).round(1)

fig_b = px.line(
    evol_tipo.sort_values('snapshot_week'),
    x='snapshot_week', y='kr1_pct', color='cycle_type',
    title='Evolução do KR1 Consolidado — Base vs Extra',
    labels={'snapshot_week': 'Semana', 'kr1_pct': 'KR1 (%)', 'cycle_type': 'Tipo de Ciclo'},
    markers=True,
    color_discrete_map={'Base': '#636EFA', 'Extra': '#EF553B'}
)
fig_b.add_hline(y=85, line_dash="dot", line_color="#636EFA", annotation_text="Meta Base (85%)")
fig_b.add_hline(y=70, line_dash="dot", line_color="#EF553B", annotation_text="Meta Extra (70%)")
fig_b.update_layout(yaxis_range=[0, 105], template='plotly_white')
fig_b.show()

In [ ]:
# --- Gráfico C: Heatmap Semana × Mês-Alvo (apenas ciclos Base) ---

evol_mes_base = evol_mes[evol_mes['cycle_type'] == 'Base'].copy()

heatmap_pivot = evol_mes_base.pivot(
    index='mes_alvo_str',
    columns='snapshot_week',
    values='kr1_pct'
)

# Ordenar linhas por mês-alvo
heatmap_pivot = heatmap_pivot.sort_index()

# Formatar colunas como string de semana
heatmap_pivot.columns = [c.strftime('%d/%m') for c in heatmap_pivot.columns]

fig_c = px.imshow(
    heatmap_pivot,
    title='KR1 (%) — Semana × Mês-Alvo [Ciclos Base]',
    labels=dict(x='Semana do Snapshot', y='Mês-Alvo', color='KR1 (%)'),
    color_continuous_scale='RdYlGn',
    zmin=50, zmax=100,
    text_auto='.0f',
    aspect='auto'
)
fig_c.update_layout(template='plotly_white')
fig_c.show()

# 7b. Evolução Temporal das Alterações Externas

Mesma lógica da seção anterior, mas olhando apenas alterações externas ao plano.
Aqui o foco é medir **% do volume original impactado por fatores externos** ao longo do tempo.

In [ ]:
# --- Gráfico D: Curva de alterações externas por mês-alvo × tipo de ciclo ---

evol_ext_mes = df_evolucao.groupby(['snapshot_week', 'mes_alvo_str', 'cycle_type']).agg(
    vol_original=('vol_original', 'sum'),
    vol_ext_any=('vol_ext_any', 'sum'),
    vol_ext_cancel=('vol_ext_cancel', 'sum'),
    vol_ext_date_rev=('vol_ext_date_rev', 'sum'),
).reset_index()

evol_ext_mes['pct_ext_any'] = (evol_ext_mes['vol_ext_any'] / evol_ext_mes['vol_original'] * 100).round(1)

fig_d = px.line(
    evol_ext_mes.sort_values('snapshot_week'),
    x='snapshot_week', y='pct_ext_any', color='mes_alvo_str',
    facet_col='cycle_type',
    title='Evolução das Alterações Externas por Mês-Alvo (Base vs Extra)',
    labels={'snapshot_week': 'Semana', 'pct_ext_any': '% Volume Impactado (EXT)', 'mes_alvo_str': 'Mês-Alvo'},
    markers=True
)
fig_d.update_layout(template='plotly_white')
fig_d.update_yaxes(rangemode='tozero')
fig_d.show()

In [ ]:
# --- Gráfico E: Alterações externas consolidadas ao longo do tempo — Base vs Extra ---

evol_ext_tipo = df_evolucao.groupby(['snapshot_week', 'cycle_type']).agg(
    vol_original=('vol_original', 'sum'),
    vol_ext_any=('vol_ext_any', 'sum'),
    vol_ext_cancel=('vol_ext_cancel', 'sum'),
    vol_ext_date_rev=('vol_ext_date_rev', 'sum'),
).reset_index()

evol_ext_tipo['pct_ext_any'] = (evol_ext_tipo['vol_ext_any'] / evol_ext_tipo['vol_original'] * 100).round(1)

fig_e = px.line(
    evol_ext_tipo.sort_values('snapshot_week'),
    x='snapshot_week', y='pct_ext_any', color='cycle_type',
    title='Evolução Consolidada das Alterações Externas — Base vs Extra',
    labels={'snapshot_week': 'Semana', 'pct_ext_any': '% Volume Impactado (EXT)', 'cycle_type': 'Tipo de Ciclo'},
    markers=True,
    color_discrete_map={'Base': '#636EFA', 'Extra': '#EF553B'}
)
fig_e.update_layout(template='plotly_white')
fig_e.update_yaxes(rangemode='tozero')
fig_e.show()

In [ ]:
# --- Gráfico F: Heatmap Semana × Mês-Alvo das alterações externas (apenas ciclos Base) ---

evol_ext_base = evol_ext_mes[evol_ext_mes['cycle_type'] == 'Base'].copy()

heatmap_ext_pivot = evol_ext_base.pivot(
    index='mes_alvo_str',
    columns='snapshot_week',
    values='pct_ext_any'
)
heatmap_ext_pivot = heatmap_ext_pivot.sort_index()
heatmap_ext_pivot.columns = [c.strftime('%d/%m') for c in heatmap_ext_pivot.columns]

fig_f = px.imshow(
    heatmap_ext_pivot,
    title='Alterações Externas (%) — Semana × Mês-Alvo [Ciclos Base]',
    labels=dict(x='Semana do Snapshot', y='Mês-Alvo', color='% Impactado (EXT)'),
    color_continuous_scale='YlOrRd',
    zmin=0,
    text_auto='.1f',
    aspect='auto'
)
fig_f.update_layout(template='plotly_white')
fig_f.show()

# 8. Export

In [ ]:
output_path = '../3_Outputs/'

# KR1 por coorte
kr1_coorte.to_csv(f'{output_path}kr1_por_coorte_{today}.csv', index=False)

# KR1 por mês-alvo
kr1_mes.to_csv(f'{output_path}kr1_por_mes_{today}.csv', index=False)

# Dados granulares (plano vs atual com flags)
df_plano.to_csv(f'{output_path}plano_vs_atual_{today}.csv', index=False)

# Frequência de revisões
df_freq.to_csv(f'{output_path}frequencia_revisoes_{today}.csv', index=False)

# Evolução temporal do KR1
df_evolucao.to_csv(f'{output_path}kr1_evolucao_{today}.csv', index=False)

print(f"Arquivos exportados em {output_path} com sufixo _{today}")

Arquivos exportados em ../3_Outputs/ com sufixo _20260422
